# Larger English → Russian scratch Transformer

The user-reported fast run achieved multi-reference greedy BLEU 54.53 / chrF 67.93 in 447 seconds, with validation loss still falling. This notebook now defaults to the **512-dimensional, six-layer large model**. Select T4 GPU. This is a NEW experiment; do not resume the fast checkpoint into a different architecture/tokenizer. Run one epoch, inspect the actual time/memory, then resume the same large checkpoint. The reported fast scores have not been independently reproduced from artifacts. No final combined notebook is included.

## 1. Clone or update the implementation branch
The default ref is `feat/scratch-transformer`. A clean existing checkout is updated with a fast-forward merge so rerunning this notebook picks up the speed fixes. A dirty checkout is preserved and raises an error; save your changes before updating. Change the ref to `main` after merge if desired. All long-running commands below stream merged stdout/stderr into the cell and propagate failures.

In [ ]:
from pathlib import Path
import subprocess
import sys

def stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

REPO_URL = "https://github.com/vcdx1111/Machine-translation-English-to-Russian-.git"
REPO_REF = "feat/scratch-transformer"
REPO = Path("/content/Machine-translation-English-to-Russian-")
if not REPO.exists():
    stream(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO)])
else:
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Preserve local changes before updating"
    stream(["git", "-C", str(REPO), "fetch", "origin", REPO_REF])
    stream(["git", "-C", str(REPO), "switch", REPO_REF])
    stream(["git", "-C", str(REPO), "merge", "--ff-only", "origin/" + REPO_REF])
%cd /content/Machine-translation-English-to-Russian-
stream(["git", "log", "-1", "--oneline"])

## 2. Install pinned packages
Install into the notebook kernel environment. If Colab asks for a runtime restart, restart and rerun from the top. The command fails immediately if installation fails.

In [ ]:
stream([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
import torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime for the fast T4 configuration"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

## 3. Drive and a new large run
The large model uses 16k train-only BPE, cap 64 (to address omitted clauses), token budget 32,768, 30 epochs and patience 5. Keep `RESUME=False` only for a NEW run; after reconnect set it to True. Logs verify AMP FP16 and report peak allocated memory. If OOM occurs, reduce the token budget in a copied YAML under a new run name; the T4 memory fit is not verified here. Avoid concurrent writers.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CONFIG = "configs/transformer_colab_large.yaml"
RUN_NAME = "scratch_large_v3_seed42"
RESUME = False
RUN = Path("/content/drive/MyDrive/en_ru_translation") / RUN_NAME
CHECKPOINTS = RUN / "checkpoints"
RESULTS = RUN / "results"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
if RESUME:
    assert (CHECKPOINTS / "last.pt").exists(), "No large checkpoint found"
else:
    assert not (CHECKPOINTS / "last.pt").exists(), "Use a new run name or set RESUME=True"

## 4. Load and audit the checked-in raw TSV
Read all three columns and create source-grouped train/validation/test splits. The same normalization and split seed are used during training. SentencePiece sees train pairs only. Inspect the measured report before training.

In [ ]:
from src.data import prepare, read_raw
RAW = REPO / "data/raw/rus.txt"
assert RAW.is_file(), "Expected the tracked data/raw/rus.txt"
rows, widths, malformed = read_raw(RAW)
print("Rows:", len(rows), "Columns:", widths, "Malformed:", malformed)
print(rows[:2])
prepare(RAW, "data/processed/full", seed=42)
import json
print(json.dumps(json.loads(Path("data/processed/full/audit.json").read_text()), ensure_ascii=False, indent=2))
del rows

## 5. Measure one real epoch (no test evaluation)
This runs epoch one with the **unchanged planned 30-epoch config**, saves `last.pt`, and stops before test decoding. Progress is streamed in real time: step/total, step ETA, loss, target tokens/sec, AMP status, epoch time and remaining training ETA. Validation computes full loss plus greedy translations on 128 fixed validation sources; no beam search occurs here. Checkpoints include optimizer/scaler/RNG state, so the next training cell continues the same run. Reconnecting runs skip this cell.

In [ ]:
def training_command():
    return [sys.executable, "-u", "-m", "src.train", "--config", CONFIG,
            "--checkpoint-dir", str(CHECKPOINTS), "--results-dir", str(RESULTS)]

if not RESUME:
    stream(training_command() + ["--stop-after-epochs", "1", "--skip-test"])
else:
    print("Resuming: keep existing epoch measurement.")

## 6. Inspect the first-epoch time projection
The original fast model measured ~36 seconds/epoch on the user's T4. For the bigger model, longer token cap, larger vocabulary, 30 epochs and Drive saves, budget roughly **60–150 minutes** initially; this is an extrapolation, not a benchmark. The JSON projects training plus full greedy, full beam-4 and full beam-4 with trigram suppression from actual epoch/validation timing. Decode cost is approximate; inspect after epoch one. No data/epochs are silently dropped.

In [ ]:
import json
estimate = json.loads((RESULTS / "runtime_estimate.json").read_text())
print(json.dumps(estimate, indent=2))
print(f"Measured epoch: {estimate['measured_epoch_seconds']/60:.2f} min")
print(f"Projected training + test: {estimate['projected_total_hours']:.2f} hours (estimate)")
print(f"Remaining training: {estimate['remaining_train_seconds']/60:.1f} min")
if not estimate['within_3_hour_target']:
    print("WARNING: measured run is outside the 3-hour target; do not treat the target as a benchmark.")

## 7. Continue the SAME run and evaluate all test sources
Training resumes epoch one's optimizer/Noam/scaler/RNG state. Large defaults: 512 dimensions, six layers, eight heads, FFN 2048, dropout .1, BPE 16k, cap 64, 32,768 padded tokens, warmup 500 and Noam factor .5. Final evaluation uses **full-test beam 4**, also records full greedy, and repeats full beam 4 with `no_repeat_ngram_size=3`. `test_bleu`/`test_chrf` and `samples.txt` always use unconstrained beam. `repetition_ablation` and `samples_no_repeat.txt` record suppression on. Do not choose the best test policy and present it as an untouched held-out estimate; use validation/manual review for future decisions.

In [ ]:
stream(training_command() + ["--resume", str(CHECKPOINTS / "last.pt")])

## 8. Optional standalone evaluation
Only rerun if evaluation was interrupted. By default the large YAML evaluates full beam 4, full greedy and full trigram-suppressed beam. Best validation-loss weights and the bundled tokenizer are used. Use another results directory for deliberately changed decoding policies.

In [ ]:
RERUN_EVALUATION = False
if RERUN_EVALUATION:
    stream([sys.executable, "-u", "-m", "src.evaluate", "--checkpoint", str(CHECKPOINTS / "last.pt"),
            "--results-dir", str(RESULTS)])

## 9. Inspect/share and compare
Send metrics, train log, samples (both suppression off/on), runtime estimate, exact YAML/commit and GPU/memory warnings. `scripts/compare_results.py` compares exact held-out identities and keeps decoding policies separate. Run Helsinki in its own notebook/run directory.

In [ ]:
import csv
metrics = json.loads((RESULTS / "metrics.json").read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print((RESULTS / "train_log.csv").read_text())
print((RESULTS / "samples.txt").read_text())
assert len(list(csv.DictReader((RESULTS / "train_log.csv").open()))) > 0
assert (RESULTS / "samples.txt").read_text().count("EN: ") == 30
print("Results saved in:", RESULTS)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)